load the libraries


In [1]:
import requests, pandas, cryptography, dotenv
print("env ok")

env ok


In [2]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("ODDS_API_KEY")  # match the variable name in your .env

resp = requests.get(
    "https://api.the-odds-api.com/v4/sports",
    params={"apiKey": API_KEY},
)

print("Status:", resp.status_code)
print("Requests remaining:", resp.headers.get("x-requests-remaining"))
print("Requests used:", resp.headers.get("x-requests-used"))

sports = resp.json()
print(f"\n{len(sports)} sports returned")
print("NBA present:", any(s["key"] == "basketball_nba" for s in sports))

Status: 200
Requests remaining: 500
Requests used: 0

45 sports returned
NBA present: False


In [4]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

project_root = Path(dotenv_path).parent          # the folder .env lives in
rel = os.getenv("KALSHI_PRIVATE_KEY_PATH")        # "pem/kalshi_key.pem"
path = project_root / rel
print("Resolved path:", path, "| exists:", path.exists())

Resolved path: /Users/dylanryan/Desktop/Summer Research/pem/kalshi_key.pem | exists: True


# Polymarket

In [5]:
import requests

r = requests.get("https://gamma-api.polymarket.com/markets",
                 params={"limit": 1}, timeout=15)
print("Polymarket status:", r.status_code)
print("Sample field keys:", list(r.json()[0].keys())[:10])

nba = requests.get("https://gamma-api.polymarket.com/events",
                   params={"tag_id": 745, "closed": "true", "limit": 3}, timeout=15)
print("NBA events returned:", len(nba.json()))
for e in nba.json():
    print("  -", e.get("title"))

Polymarket status: 200
Sample field keys: ['id', 'question', 'conditionId', 'slug', 'resolutionSource', 'endDate', 'liquidity', 'startDate', 'image', 'icon']
NBA events returned: 3
  - NBA Play-In: Lakers vs Pelicans
  - NBA Play-In: Warriors vs Kings
  - NBA Play-In: Heat vs 76ers


In [6]:
import requests

for base in ["https://api.elections.kalshi.com/trade-api/v2",
             "https://external-api.kalshi.com/trade-api/v2",
             "https://api.kalshi.com/trade-api/v2"]:
    try:
        r = requests.get(f"{base}/markets", params={"limit": 1}, timeout=15)
        print(f"{base} -> {r.status_code}")
        if r.status_code == 200:
            print("  sample ticker:", r.json().get("markets", [{}])[0].get("ticker"))
            print("  WORKING BASE:", base)
            break
    except Exception as ex:
        print(f"{base} -> ERROR {ex}")

https://api.elections.kalshi.com/trade-api/v2 -> 200
  sample ticker: KXMVESPORTSMULTIGAMEEXTENDED-S202671E0CC0C468-F0B19F3D3CC
  WORKING BASE: https://api.elections.kalshi.com/trade-api/v2


In [10]:
import os, requests
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
PH_KEY = os.getenv("PREDICTION_HUNT_API_KEY")

r = requests.get("https://www.predictionhunt.com/api/v2/events",
                 headers={"X-API-Key": PH_KEY}, timeout=45)
print("status:", r.status_code)
print(r.text[:500])

status: 500
Internal Server Error


In [ ]:
from collections import Counter

# Drop any non-dict junk that slipped into the list
clean = [e for e in events if isinstance(e, dict)]
print(f"Clean event dicts: {len(clean)} of {len(events)}")

dates = sorted(e.get("startDate", "")[:10] for e in clean if e.get("startDate"))
print("Earliest:", dates[0], " Latest:", dates[-1])

def season_of(d):
    y, m = int(d[:4]), int(d[5:7])
    start = y if m >= 9 else y - 1
    return f"{start}-{str(start+1)[-2:]}"

print("By season:", dict(sorted(Counter(season_of(d) for d in dates).items())))
print("Markets-per-event distribution:",
      dict(sorted(Counter(len(e.get("markets", [])) for e in clean).items())))

Clean event dicts: 2100 of 2102
Earliest: 2024-04-15  Latest: 2025-10-23
By season: {'2023-24': 96, '2024-25': 1869, '2025-26': 135}
Markets-per-event distribution: {1: 1605, 2: 14, 3: 21, 4: 18, 5: 24, 6: 9, 7: 31, 8: 27, 9: 7, 10: 9, 11: 8, 12: 5, 13: 2, 14: 5, 15: 7, 16: 12, 17: 8, 18: 4, 19: 4, 20: 3, 21: 4, 22: 14, 23: 227, 24: 2, 25: 1, 28: 2, 29: 1, 30: 11, 31: 1, 33: 1, 35: 1, 36: 1, 56: 5, 57: 1, 63: 1, 64: 1, 69: 1, 70: 1, 73: 1}


In [15]:
import requests
from collections import Counter

BASE = "https://gamma-api.polymarket.com"
NBA_TAG = 745

events, offset, limit = [], 0, 100
while True:
    r = requests.get(f"{BASE}/events",
                     params={"tag_id": NBA_TAG, "closed": "true",
                             "limit": limit, "offset": offset,
                             "order": "endDate", "ascending": "false"},  # newest settled first
                     timeout=30)
    page = r.json()
    if not isinstance(page, list) or not page:
        break
    events.extend([e for e in page if isinstance(e, dict)])
    if len(page) < limit:
        break
    offset += limit
    if offset >= 4000:   # safety stop; we'll see if we even approach it
        break

print(f"Pulled: {len(events)}")
dates = sorted(e.get("startDate","")[:10] for e in events if e.get("startDate"))
print("Earliest:", dates[0], " Latest:", dates[-1])

def season_of(d):
    y, m = int(d[:4]), int(d[5:7])
    s = y if m >= 9 else y - 1
    return f"{s}-{str(s+1)[-2:]}"
print("By season:", dict(sorted(Counter(season_of(d) for d in dates).items())))

Pulled: 2100
Earliest: 2024-09-24  Latest: 2026-06-23
By season: {'2024-25': 519, '2025-26': 1581}


In [17]:
from collections import Counter

def season_of(d):
    y, m = int(d[:4]), int(d[5:7]); s = y if m >= 9 else y - 1
    return f"{s}-{str(s+1)[-2:]}"

s2526 = [e for e in events
         if e.get("startDate") and season_of(e["startDate"][:10]) == "2025-26"]
print(f"2025-26 events: {len(s2526)}")
print("Markets-per-event:",
      dict(sorted(Counter(len(e.get('markets', [])) for e in s2526).items())))

# Peek inside the biggest cluster of multi-market events
sizes = Counter(len(e.get('markets', [])) for e in s2526)
top_multi = sizes.most_common()
print("\nMost common event sizes (n_markets: count):", top_multi[:6])

# Show one event from the largest non-trivial cluster, and one single-market event
multi_size = next(n for n, _ in top_multi if n > 1)
ex = next(e for e in s2526 if len(e.get('markets', [])) == multi_size)
print(f"\n=== {multi_size}-MARKET EVENT ===")
print("title:", ex.get("title"), "| start:", ex.get("startDate","")[:10])
for m in ex.get("markets", []):
    print("   •", m.get("question"))

s1 = next(e for e in s2526 if len(e.get('markets', [])) == 1)
print("\n=== 1-MARKET EVENT ===")
print("title:", s1.get("title"))
print("   •", s1["markets"][0].get("question"))

2025-26 events: 1581
Markets-per-event: {1: 102, 3: 23, 4: 5, 5: 26, 6: 12, 7: 15, 8: 11, 9: 16, 10: 24, 11: 26, 12: 56, 13: 47, 14: 46, 15: 36, 16: 31, 17: 23, 18: 11, 19: 15, 20: 10, 21: 9, 22: 2, 23: 3, 24: 1, 25: 1, 26: 4, 28: 8, 30: 6, 31: 7, 32: 2, 33: 16, 36: 9, 37: 8, 38: 10, 39: 30, 40: 52, 41: 76, 42: 102, 43: 120, 44: 84, 45: 78, 46: 58, 47: 44, 48: 36, 49: 32, 50: 13, 51: 19, 52: 15, 53: 5, 54: 5, 55: 10, 56: 11, 57: 4, 58: 5, 59: 2, 60: 5, 61: 8, 62: 6, 63: 4, 64: 4, 65: 12, 66: 11, 67: 11, 68: 10, 69: 7, 70: 10, 71: 8, 72: 8, 73: 6, 74: 4, 75: 7, 76: 7, 77: 11, 78: 4, 79: 3, 80: 1, 81: 2, 82: 1, 83: 2, 84: 1, 97: 1, 116: 1, 118: 2, 119: 2}

Most common event sizes (n_markets: count): [(43, 120), (1, 102), (42, 102), (44, 84), (45, 78), (41, 76)]

=== 43-MARKET EVENT ===
title: 2026 NBA Draft: 1st Overall pick | start: 2026-01-02
   • Will Cameron Boozer be the first pick in the 2026 NBA draft?
   • Will Caleb Wilson be the first pick in the 2026 NBA draft?
   • Will Jayde

In [18]:
import re

def season_of(d):
    y, m = int(d[:4]), int(d[5:7]); s = y if m >= 9 else y - 1
    return f"{s}-{str(s+1)[-2:]}"

s2526 = [e for e in events
         if e.get("startDate") and season_of(e["startDate"][:10]) == "2025-26"]

# Game events: title contains "vs" or "@" between two teams
def looks_like_game(title):
    t = (title or "").lower()
    return (" vs " in t or " vs. " in t or " @ " in t) and "draft" not in t

games = [e for e in s2526 if looks_like_game(e.get("title"))]
futures = [e for e in s2526 if not looks_like_game(e.get("title"))]
print(f"Game-like events: {len(games)} | Non-game/futures: {len(futures)}")

# Distribution of markets-per-event AMONG games only
from collections import Counter
print("Markets-per-game distribution:",
      dict(sorted(Counter(len(g.get('markets', [])) for g in games).items())))

# Dump a few games of different sizes to see their market structure
shown = set()
for g in sorted(games, key=lambda e: len(e.get('markets', []))):
    n = len(g.get('markets', []))
    if n in shown:
        continue
    shown.add(n)
    print(f"\n=== GAME, {n} market(s): {g.get('title')} ({g.get('startDate','')[:10]}) ===")
    for m in g.get('markets', [])[:12]:
        print("   •", m.get("question"))
    if len(shown) >= 5:
        break

Game-like events: 1455 | Non-game/futures: 126
Markets-per-game distribution: {1: 72, 3: 23, 4: 3, 5: 17, 6: 9, 7: 15, 8: 10, 9: 14, 10: 24, 11: 26, 12: 56, 13: 47, 14: 43, 15: 36, 16: 29, 17: 22, 18: 11, 19: 13, 20: 5, 21: 6, 22: 1, 24: 1, 25: 1, 28: 7, 31: 2, 33: 1, 36: 4, 37: 8, 38: 10, 39: 29, 40: 50, 41: 76, 42: 102, 43: 119, 44: 84, 45: 78, 46: 58, 47: 44, 48: 35, 49: 31, 50: 13, 51: 19, 52: 14, 53: 5, 54: 5, 55: 10, 56: 6, 57: 3, 58: 5, 59: 2, 60: 5, 61: 8, 62: 6, 63: 3, 64: 3, 65: 12, 66: 11, 67: 11, 68: 10, 69: 6, 70: 9, 71: 8, 72: 8, 73: 5, 74: 4, 75: 7, 76: 7, 77: 11, 78: 3, 79: 3, 81: 2, 82: 1, 83: 2, 84: 1, 116: 1, 118: 2, 119: 2}

=== GAME, 1 market(s): NBA Playoffs: 76ers vs. Celtics Total Games O/U 4.5 (2026-04-16) ===
   • NBA Playoffs: 76ers vs. Celtics Total Games O/U 4.5

=== GAME, 3 market(s): Knicks vs. Spurs (2026-06-13) ===
   • Knicks vs. Spurs: Team to Score First
   • Knicks vs. Spurs
   • Knicks vs. Spurs: Odd/Even Score

=== GAME, 4 market(s): Warriors vs. 

In [19]:
import re

def classify(q):
    ql = (q or "").lower()
    if re.search(r"spread", ql):                 return "spread"
    if re.search(r"\bo/u\b|over|under|total", ql): return "total"
    if re.search(r" vs\.? | @ ", ql) and ":" not in ql: return "moneyline"
    return "prop/other"

from collections import Counter
spread_depth = Counter()
type_presence = Counter()
for g in games:
    qs = [m.get("question") for m in g.get("markets", [])]
    kinds = [classify(q) for q in qs]
    n_spread = kinds.count("spread")
    spread_depth[n_spread] += 1
    for k in set(kinds):
        type_presence[k] += 1

print("Games by # of spread contracts:", dict(sorted(spread_depth.items())))
print("\nGames containing each market type:", dict(type_presence))
print(f"\nGames with a real ladder (3+ spreads): "
      f"{sum(v for k,v in spread_depth.items() if k >= 3)}")

Games by # of spread contracts: {0: 89, 1: 17, 2: 67, 3: 168, 4: 316, 5: 303, 6: 156, 7: 88, 8: 48, 9: 17, 10: 22, 11: 9, 12: 9, 13: 14, 14: 12, 15: 13, 16: 13, 17: 16, 18: 15, 19: 13, 20: 5, 21: 10, 22: 9, 23: 3, 24: 7, 25: 2, 26: 3, 27: 3, 28: 3, 29: 1, 50: 4}

Games containing each market type: {'moneyline': 1307, 'prop/other': 1233, 'total': 1395, 'spread': 1366}

Games with a real ladder (3+ spreads): 1282


# kalshi

In [20]:
import requests

BASE = "https://api.elections.kalshi.com/trade-api/v2"

def fetch_all_series(params):
    out, cursor = [], None
    while True:
        p = dict(params)
        if cursor:
            p["cursor"] = cursor
        r = requests.get(f"{BASE}/series", params=p, timeout=30)
        if r.status_code != 200:
            return r.status_code, out, r.text[:300]
        data = r.json()
        out.extend(data.get("series", []))
        cursor = data.get("cursor")
        if not cursor:
            return 200, out, None

# NBA sits under the "Sports" category
status, series, err = fetch_all_series({"category": "Sports"})
print("status:", status, "| sports series:", len(series))
if not series:                       # category value may differ — fall back to listing all
    status, series, err = fetch_all_series({})
    print("fallback (no filter) | total series:", len(series), "| err:", err)

def is_nba(s):
    blob = (s.get("ticker","") + " " + s.get("title","") + " " + str(s.get("tags",""))).upper()
    return "NBA" in blob or "BASKETBALL" in blob

nba = [s for s in series if is_nba(s)]
print(f"\nNBA-related series ({len(nba)}):")
for s in nba:
    print(f"   {s.get('ticker'):26} | {s.get('title')}")

status: 200 | sports series: 2283

NBA-related series (463):
   KXNBAFIRSTBASKET           | Pro Basketball First Basket
   KXNBAUPSET                 | NBA Number of Upsets per Round
   KXNBAMVPDPOY               | Pro Basketball Player to Win MVP and DPOY
   KXNCAAMBBIGEAST            | Big East Conference Tournament
   KXNBASPORTSMANSHIP         | NBA Sportsmanship Award
   KXWNBAWINS                 | WNBA Win Totals
   KXEUROCUPGAME              | EuroCup Basketball Game
   KXNBAGAME                  | Pro Basketball Game
   KXEUROLEAGUE               | EuroLeague Champion
   KXNEXTCOACHOUTNBA          | Next coach out NBA
   KXNCAABBSPREAD             | College Baseball Spread
   KXNCAAWBGAME               | College Basketball Women's Game
   KXFIBAGAME                 | FIBA Game
   KXNCAAMBUSA                | Conference USA Conference Tournament
   KXNBAREB                   | Pro Basketball Player Rebounds
   KXBBSERIEBGAME             | Italy Serie B Basketball Game
   KXNCA

In [21]:
import requests
BASE = "https://api.elections.kalshi.com/trade-api/v2"

# 1) Confirm KXNBAGAME structure — pull a page of its markets (any status)
r = requests.get(f"{BASE}/markets",
                 params={"series_ticker": "KXNBAGAME", "limit": 5, "status": "settled"},
                 timeout=30)
print("KXNBAGAME settled (live endpoint) status:", r.status_code)
mk = r.json().get("markets", [])
print("settled markets on live endpoint:", len(mk))
for m in mk[:5]:
    print(f"   {m.get('ticker')}  | {m.get('title')}  | close={m.get('close_time','')[:10]}  result={m.get('result')}")

# 2) Find the live/historical boundary
c = requests.get(f"{BASE}/historical/cutoff", timeout=30)
print("\n/historical/cutoff status:", c.status_code)
print("   ", c.text[:200])

KXNBAGAME settled (live endpoint) status: 200
settled markets on live endpoint: 5
   KXNBAGAME-26JUN13NYKSAS-SAS  | Game 5: New York at San Antonio Winner?  | close=2026-06-14  result=no
   KXNBAGAME-26JUN13NYKSAS-NYK  | Game 5: New York at San Antonio Winner?  | close=2026-06-14  result=yes
   KXNBAGAME-26JUN10SASNYK-SAS  | Game 4: San Antonio at New York Winner?  | close=2026-06-11  result=no
   KXNBAGAME-26JUN10SASNYK-NYK  | Game 4: San Antonio at New York Winner?  | close=2026-06-11  result=yes
   KXNBAGAME-26JUN08SASNYK-SAS  | Game 3: San Antonio at New York Winner?  | close=2026-06-09  result=yes

/historical/cutoff status: 200
    {"market_settled_ts":"2026-05-01T00:00:00Z","orders_updated_ts":"2026-05-01T00:00:00Z","trades_created_ts":"2026-05-01T00:00:00Z"}


In [23]:
import re

NBA = {"ATL","BOS","BKN","CHA","CHI","CLE","DAL","DEN","DET","GSW","HOU","IND",
       "LAC","LAL","MEM","MIA","MIL","MIN","NOP","NYK","OKC","ORL","PHI","PHX",
       "POR","SAC","SAS","TOR","UTA","WAS"}

def parse_ticker(t):
    # KXNBAGAME-<DATE><AWAYHOME>-<SIDE>
    m = re.match(r"KXNBAGAME-(\d{2}[A-Z]{3}\d{2})([A-Z]+)-([A-Z]+)$", t)
    if not m:
        return None
    date, pair, side = m.groups()
    # side is a known team; the other team is pair with side removed from the correct end
    if pair.endswith(side):
        away, home = pair[:-len(side)], side          # home listed second
    elif pair.startswith(side):
        away, home = side, pair[len(side):]            # away listed first
    else:
        return ("BAD", date, pair, side)
    if away in NBA and home in NBA:
        return (date, away, home)
    return ("BAD", date, pair, side)

parsed = [parse_ticker(m["ticker"]) for m in allmk]
good  = [p for p in parsed if p and p[0] != "BAD"]
bad   = [p for p in parsed if p and p[0] == "BAD"]

games = {(d, a, h) for d, a, h in good}
print(f"Cleanly resolved game-rows: {len(good)} | unresolved: {len(bad)}")
print(f"Unique settled NBA games (validated teams): {len(games)}")
print("sample:", list(games)[:5])
print("examples that didn't resolve:", [b[1:] for b in bad[:8]])

Cleanly resolved game-rows: 2936 | unresolved: 2
Unique settled NBA games (validated teams): 1448
sample: [('26MAR28', 'PHI', 'CHA'), ('26JUN10', 'SAS', 'NYK'), ('26FEB22', 'CHA', 'WAS'), ('25NOV01', 'HOU', 'BOS'), ('26MAY19', 'CLE', 'NYK')]
examples that didn't resolve: [('25OCT13', 'GUAMIN', 'MIN'), ('25OCT13', 'GUAMIN', 'GUA')]


In [24]:
import re

# Full/nickname -> tricode (covers the 30 teams; add aliases if any slip through)
NAME2CODE = {
    "hawks":"ATL","celtics":"BOS","nets":"BKN","hornets":"CHA","bulls":"CHI",
    "cavaliers":"CLE","mavericks":"DAL","nuggets":"DEN","pistons":"DET","warriors":"GSW",
    "rockets":"HOU","pacers":"IND","clippers":"LAC","lakers":"LAL","grizzlies":"MEM",
    "heat":"MIA","bucks":"MIL","timberwolves":"MIN","pelicans":"NOP","knicks":"NYK",
    "thunder":"OKC","magic":"ORL","76ers":"PHI","sixers":"PHI","suns":"PHX",
    "trail blazers":"POR","blazers":"POR","kings":"SAC","spurs":"SAS","raptors":"TOR",
    "jazz":"UTA","wizards":"WAS",
}

def pm_teams(title):
    # split "A vs. B" / "A @ B" and map each side to a tricode
    parts = re.split(r"\s+vs\.?\s+|\s+@\s+", (title or "").lower())
    if len(parts) != 2:
        return None
    codes = []
    for side in parts:
        hit = next((c for name, c in NAME2CODE.items() if name in side), None)
        if not hit:
            return None
        codes.append(hit)
    return tuple(codes)  # (left, right) as written in the title

# Build Polymarket game keys (date + unordered team pair)
def iso_to_kalshi_date(d):  # 2026-06-13 -> 26JUN13
    import datetime as dt
    return dt.datetime.strptime(d[:10], "%Y-%m-%d").strftime("%y%b%d").upper()

pm_games = {}
for e in games_pm if (games_pm := [g for g in events
        if g.get("startDate") and season_of(g["startDate"][:10])=="2025-26"
        and (" vs" in (g.get("title","").lower()) or " @ " in (g.get("title","").lower()))]) else []:
    tc = pm_teams(e.get("title"))
    if tc:
        key = (iso_to_kalshi_date(e["startDate"]), frozenset(tc))
        pm_games[key] = e

# Kalshi keys (order-independent)
ks_games = {(d, frozenset((a, h))) for d, a, h in good}

pm_keys = set(pm_games)
overlap = pm_keys & ks_games
print(f"Polymarket game keys: {len(pm_keys)}")
print(f"Kalshi game keys:     {len(ks_games)}")
print(f"MATCHED (both platforms): {len(overlap)}")
print(f"Polymarket-only: {len(pm_keys - ks_games)} | Kalshi-only: {len(ks_games - pm_keys)}")
print("\nsample matches:", list(overlap)[:3])

Polymarket game keys: 1405
Kalshi game keys:     1448
MATCHED (both platforms): 51
Polymarket-only: 1354 | Kalshi-only: 1397

sample matches: [('26MAY07', frozenset({'CLE', 'DET'})), ('26MAY24', frozenset({'SAS', 'OKC'})), ('26JUN13', frozenset({'NYK', 'SAS'}))]


In [25]:
import datetime as dt

def iso_date(d):
    return dt.datetime.strptime(d[:10], "%Y-%m-%d").date()

def kalshi_date(code):           # 26JUN13 -> date(2026,6,13)
    return dt.datetime.strptime(code, "%y%b%d").date()

# Index Kalshi games by team-pair -> set of dates
from collections import defaultdict
ks_by_pair = defaultdict(set)
for d, a, h in good:
    ks_by_pair[frozenset((a, h))].add(kalshi_date(d))

# Rebuild PM games with real dates + team pair
pm_list = []
for e in events:
    if not (e.get("startDate") and season_of(e["startDate"][:10]) == "2025-26"):
        continue
    title = e.get("title","").lower()
    if not (" vs" in title or " @ " in title):
        continue
    tc = pm_teams(e.get("title"))
    if tc:
        pm_list.append((iso_date(e["startDate"]), frozenset(tc), e))

# Match: same teams, date within +/- 1 day
matched, pm_only = [], []
for d, pair, e in pm_list:
    ks_dates = ks_by_pair.get(pair, set())
    if any(abs((d - kd).days) <= 1 for kd in ks_dates):
        matched.append((d, pair))
    else:
        pm_only.append((d, pair))

print(f"PM game events parsed:  {len(pm_list)}")
print(f"MATCHED (±1 day):       {len(matched)}")
print(f"PM-only (no Kalshi):    {len(pm_only)}")

# Sanity: how far apart are the matched dates, really? (confirms the UTC shift story)
from collections import Counter
offsets = []
for d, pair, e in pm_list:
    for kd in ks_by_pair.get(pair, set()):
        if abs((d - kd).days) <= 1:
            offsets.append((d - kd).days); break
print("date-offset distribution (PM minus Kalshi):", dict(Counter(offsets)))

PM game events parsed:  1452
MATCHED (±1 day):       155
PM-only (no Kalshi):    1297
date-offset distribution (PM minus Kalshi): {0: 61, -1: 67, 1: 27}


In [26]:
# What fraction of PM titles even parse to two teams?
parsed_ct = sum(1 for d, pair, e in pm_list)
none_ct = 0
sample_unparsed = []
for e in events:
    if not (e.get("startDate") and season_of(e["startDate"][:10]) == "2025-26"):
        continue
    title = e.get("title","")
    tl = title.lower()
    if not (" vs" in tl or " @ " in tl):
        continue
    if pm_teams(title) is None:
        none_ct += 1
        if len(sample_unparsed) < 15:
            sample_unparsed.append(title)

print(f"PM game-like titles that FAILED to parse to 2 teams: {none_ct}")
print("examples of unparsed titles:")
for t in sample_unparsed:
    print("   ", t)

# Compare the actual team-pair vocabularies (ignore dates entirely)
pm_pairs = {pair for d, pair, e in pm_list}
ks_pairs = set(ks_by_pair.keys())
print(f"\nDistinct PM team-pairs: {len(pm_pairs)} | distinct Kalshi team-pairs: {len(ks_pairs)}")
print(f"Team-pairs in BOTH (dateless): {len(pm_pairs & ks_pairs)}")
print(f"Pairs only in PM: {len(pm_pairs - ks_pairs)}")
print("  sample PM-only pairs:", [tuple(p) for p in list(pm_pairs - ks_pairs)[:10]])
print("  sample Kalshi pairs :", [tuple(p) for p in list(ks_pairs)[:10]])

PM game-like titles that FAILED to parse to 2 teams: 3
examples of unparsed titles:
    Team USA Stars vs. Team USA Stripes
    Team World vs. Team USA Stripes
    Team World vs. Team USA Stars

Distinct PM team-pairs: 407 | distinct Kalshi team-pairs: 435
Team-pairs in BOTH (dateless): 406
Pairs only in PM: 1
  sample PM-only pairs: [('BKN',)]
  sample Kalshi pairs : [('DEN', 'OKC'), ('UTA', 'BOS'), ('MIL', 'MIA'), ('SAS', 'LAC'), ('CHA', 'SAS'), ('DEN', 'NOP'), ('UTA', 'NOP'), ('ORL', 'CHA'), ('ORL', 'LAC'), ('MIA', 'HOU')]


In [27]:
import datetime as dt
from collections import Counter, defaultdict

# For each PM game, find Kalshi games with the SAME team-pair, and record the smallest date gap
ks_by_pair = defaultdict(list)
for d, a, h in good:
    ks_by_pair[frozenset((a, h))].append(dt.datetime.strptime(d, "%y%b%d").date())

gaps = []
no_pair = 0
for d, pair, e in pm_list:
    kdates = ks_by_pair.get(pair, [])
    if not kdates:
        no_pair += 1
        continue
    gaps.append(min((d - kd).days for kd in kdates if abs((d - kd).days) <= 5) 
                if any(abs((d-kd).days) <= 5 for kd in kdates) else "none<=5")

print("PM games with no matching team-pair at all:", no_pair)
print("Nearest-date gap distribution (PM minus Kalshi, days):")
print("  ", dict(Counter(gaps)))

PM games with no matching team-pair at all: 3
Nearest-date gap distribution (PM minus Kalshi, days):
   {0: 17, -3: 49, -4: 129, -5: 96, -2: 36, 1: 11, -1: 24, 'none<=5': 1040, 2: 10, 4: 13, 5: 15, 3: 9}


In [28]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd

# Pull all 2025-26 NBA games (regular season + playoffs)
gf = leaguegamefinder.LeagueGameFinder(
    season_nullable="2025-26",
    league_id_nullable="00"          # 00 = NBA
)
games_df = gf.get_data_frames()[0]
print("Raw rows:", len(games_df))   # 2 rows per game (one per team)
print(games_df.columns.tolist())
print(games_df[["GAME_ID","GAME_DATE","MATCHUP","TEAM_ABBREVIATION","WL","PTS"]].head(6))

# Collapse to one row per game: GAME_ID is the canonical key
print("\nUnique GAME_IDs:", games_df["GAME_ID"].nunique())
print("Date range:", games_df["GAME_DATE"].min(), "->", games_df["GAME_DATE"].max())

Raw rows: 2805
['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
      GAME_ID   GAME_DATE      MATCHUP TEAM_ABBREVIATION WL  PTS
0  0042500405  2026-06-13  SAS vs. NYK               SAS  L   90
1  0042500405  2026-06-13    NYK @ SAS               NYK  W   94
2  0042500404  2026-06-10  NYK vs. SAS               NYK  W  107
3  0042500404  2026-06-10    SAS @ NYK               SAS  L  106
4  0042500403  2026-06-08    SAS @ NYK               SAS  W  115
5  0042500403  2026-06-08  NYK vs. SAS               NYK  L  111

Unique GAME_IDs: 1401
Date range: 2025-10-02 -> 2026-06-13


In [29]:
import datetime as dt

# Collapse two team-rows into one game record using MATCHUP home/away
game_rows = {}
for _, r in games_df.iterrows():
    gid = r["GAME_ID"]
    if gid not in game_rows:
        game_rows[gid] = {}
    if " vs. " in r["MATCHUP"]:        # this team is HOME
        game_rows[gid]["home"] = r["TEAM_ABBREVIATION"]
        game_rows[gid]["home_pts"] = r["PTS"]
    elif " @ " in r["MATCHUP"]:         # this team is AWAY
        game_rows[gid]["away"] = r["TEAM_ABBREVIATION"]
        game_rows[gid]["away_pts"] = r["PTS"]
    game_rows[gid]["date"] = dt.datetime.strptime(r["GAME_DATE"], "%Y-%m-%d").date()

# Game type from GAME_ID prefix
def gtype(gid):
    return {"001":"preseason","002":"regular","003":"allstar","004":"playoff"}.get(gid[:3], "other")

from collections import Counter
print("Games by type:", dict(Counter(gtype(g) for g in game_rows)))

# Build the canonical spine: keep regular + playoff only
spine = {}
for gid, g in game_rows.items():
    if gtype(gid) in ("regular", "playoff") and "home" in g and "away" in g:
        spine[gid] = {**g, "pair": frozenset((g["home"], g["away"])), "type": gtype(gid)}

print("Spine games (regular + playoff):", len(spine))
print("Date range:", min(g["date"] for g in spine.values()),
      "->", max(g["date"] for g in spine.values()))
# sample
for gid in list(spine)[:3]:
    g = spine[gid]
    print(f"   {gid} {g['date']} {g['away']}@{g['home']} {g['away_pts']}-{g['home_pts']}")

Games by type: {'playoff': 85, 'other': 7, 'regular': 1230, 'allstar': 8, 'preseason': 71}
Spine games (regular + playoff): 1310
Date range: 2025-10-21 -> 2026-06-13
   0042500405 2026-06-13 NYK@SAS 94-90
   0042500404 2026-06-10 SAS@NYK 106-107
   0042500403 2026-06-08 SAS@NYK 115-111


In [30]:
import datetime as dt
from collections import Counter, defaultdict

# Index the spine by team-pair -> list of (date, gid)
spine_by_pair = defaultdict(list)
for gid, g in spine.items():
    spine_by_pair[g["pair"]].append((g["date"], gid))

pm_matched, pm_unmatched, offsets = {}, [], []
for d, pair, e in pm_list:                      # pm_list: (pm_date, pair, event) from before
    cands = spine_by_pair.get(pair, [])
    if not cands:
        pm_unmatched.append((d, pair, e.get("title")))
        continue
    # nearest spine game of this matchup
    best_gid, best_gap = min(((gid, (d - sd).days) for sd, gid in cands),
                             key=lambda x: abs(x[1]))
    if abs(best_gap) <= 2:
        pm_matched[best_gid] = e
        offsets.append(best_gap)
    else:
        pm_unmatched.append((d, pair, e.get("title")))

print(f"PM games matched to spine: {len(pm_matched)} of {len(spine)} spine games")
print(f"PM games NOT matched:      {len(pm_unmatched)}")
print(f"Date-offset (PM minus official, days): {dict(Counter(offsets))}")
print("\nsample unmatched PM titles:", [t for _,_,t in pm_unmatched[:8]])

PM games matched to spine: 130 of 1310 spine games
PM games NOT matched:      1270
Date-offset (PM minus official, days): {0: 62, -1: 65, -2: 25, 1: 20, 2: 10}

sample unmatched PM titles: ['Knicks vs. Spurs', 'Knicks vs. Spurs', 'NBA Playoffs: 76ers vs. Celtics Total Games O/U 4.5', 'NBA Playoffs: Rockets vs. Lakers Total Games O/U 5.5', 'NBA Playoffs: Raptors vs. Cavaliers Total Games O/U 5.5', 'NBA Playoffs: Timberwolves vs. Nuggets Total Games O/U 5.5', 'NBA Playoffs: Thunder vs. Lakers Total Games O/U 5.5', 'NBA Playoffs: Who Will Win Series? - Spurs vs. Timberwolves ']


In [31]:
import requests, json

# Pull markets (not events) for a known Finals matchup, via the markets endpoint,
# and dump the sports-relevant fields the docs say exist.
r = requests.get("https://gamma-api.polymarket.com/markets",
                 params={"tag_id": 745, "closed": "true", "limit": 100,
                         "order": "endDate", "ascending": "false"}, timeout=30)
mkts = r.json()
print("markets pulled:", len(mkts))
print("FIELD KEYS present on a market:", sorted(mkts[0].keys()))

# Look for game_id / sports type / start-time style fields on a few
probe = ["gameId","game_id","gameStartTime","sportsMarketType","slug","question",
         "startDate","endDate","umaResolutionStatus"]
print("\nSample markets:")
for m in mkts[:8]:
    row = {k: m.get(k) for k in probe if k in m}
    print("  ", row)

markets pulled: 100
FIELD KEYS present on a market: ['acceptingOrders', 'acceptingOrdersTimestamp', 'active', 'approved', 'archived', 'automaticallyActive', 'automaticallyResolved', 'bestAsk', 'clearBookOnStart', 'clobTokenIds', 'closed', 'closedTime', 'comboStatus', 'competitive', 'conditionId', 'createdAt', 'customLiveness', 'cyom', 'deploying', 'deployingTimestamp', 'description', 'enableOrderBook', 'endDate', 'endDateIso', 'events', 'featured', 'feeSchedule', 'feeType', 'feesEnabled', 'funded', 'groupItemThreshold', 'groupItemTitle', 'hasReviewedDates', 'holdingRewardsEnabled', 'icon', 'id', 'image', 'lastTradePrice', 'liquidity', 'liquidityAmm', 'liquidityClob', 'liquidityNum', 'makerBaseFee', 'manualActivation', 'marketMakerAddress', 'negRisk', 'negRiskMarketID', 'negRiskOther', 'negRiskRequestID', 'new', 'oneDayPriceChange', 'oneWeekPriceChange', 'orderMinSize', 'orderPriceMinTickSize', 'outcomePrices', 'outcomes', 'pagerDutyNotificationEnabled', 'pendingDeployment', 'question',